## Config — channel family


In [ ]:
# nemotron_layer18_mp is the default model; set to "peav" for the PE-AV family.
FAMILY = "nemotron_layer18_mp"


## Data & setup


In [ ]:
%matplotlib inline
import sys
from pathlib import Path
import json
import shutil

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score
from IPython.display import display

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "paths.py").exists())
sys.path.insert(0, str(REPO_ROOT))
from paths import OUTPUTS

SELECTION_ROOT = (
    OUTPUTS / "cluster" / FAMILY / "_channel_timeseries_model_selection" / "norm-zscore_raw"
)
ENTITY_SINGULAR = "channel"
ENTITY_PLURAL = "channels"
LABEL_FILENAME = "channel_labels.npy"
N_ELEMENTS = int(np.load(pd.read_csv(SELECTION_ROOT / "selected_reducers.csv").iloc[0]["embedding"], mmap_mode="r").shape[0])

FIGURE_DIR = SELECTION_ROOT / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
for stale in FIGURE_DIR.glob("*.png"):
    stale.unlink()


def saved_to(path):
    print(f"Saved to {Path(path).as_uri()}")


REDUCTION_ORDER = ["pca", "mds", "isomap", "tsne", "fastica", "umap"]
DISPLAY_NAMES = {"pca": "PCA", "mds": "MDS", "isomap": "Isomap", "tsne": "t-SNE",
                  "fastica": "FastICA", "umap": "UMAP"}
CLUSTER_DISPLAY = {"kmeans": "K-means", "hdbscan": "HDBSCAN", "birch": "BIRCH"}
CLUSTER_ORDER = ["kmeans", "hdbscan", "birch"]
ROLE_NAMES = {"display_2d": "selected 2D", "display_3d": "selected 3D",
              "latent_best": "selected latent-best"}

reduction_sweep = pd.read_csv(SELECTION_ROOT / "reduction_sweep.csv")
selected_reducers = pd.read_csv(SELECTION_ROOT / "selected_reducers.csv")
selected_clusterings = pd.read_csv(SELECTION_ROOT / "selected_clusterings.csv").rename(
    columns={"selection" + "_score": "cluster_composite_score"}
)
dim_sweep = pd.read_csv(SELECTION_ROOT / "dimension_selection_sweep.csv")
latent_summary = pd.read_csv(SELECTION_ROOT / "selected_latent_dimensions.csv")
cluster_sweep = pd.read_csv(SELECTION_ROOT / "clustering_sweep.csv")
map_manifest = json.loads((SELECTION_ROOT / "selected_maps_manifest.json").read_text())

assert set(REDUCTION_ORDER) == set(selected_reducers["method"])
for role in ROLE_NAMES:
    assert selected_reducers["selection_role"].str.contains(role).sum() == 6

expected_combos = {
    (method, role, clusterer)
    for method in REDUCTION_ORDER
    for role in ROLE_NAMES
    for clusterer in CLUSTER_ORDER
}
observed_combos = set(zip(
    selected_clusterings["reduction_method"],
    selected_clusterings["selection_role"],
    selected_clusterings["method"],
))
missing_combos = sorted(expected_combos - observed_combos)

print(f"Elements: {N_ELEMENTS:,} {ENTITY_PLURAL}")
print(f"Valid reducer/role/clusterer combinations: {len(observed_combos)}/{len(expected_combos)}")
if missing_combos:
    print("Missing combinations (no valid clustering candidate):")
    for method, role, clusterer in missing_combos:
        print(f"  - {method} / {role} / {clusterer}")
saved_to(FIGURE_DIR)

display(selected_reducers[[
    "method", "selection_role", "n_components", "quality_score",
    "dimension_criterion", "dimension_criterion_value",
]].sort_values(["method", "selection_role"]).reset_index(drop=True))


## Reducer hyperparameter and dimension selection



In [ ]:

HYPERPARAM_MEANING = {
    "n_landmarks": "landmarks fitted directly",
    "extension_neighbors": "neighbors used for out-of-sample extension",
    "mds_max_iterations": "SMACOF iterations",
    "isomap_neighbors": "neighbors in the geodesic graph",
    "tsne_perplexity": "t-SNE neighborhood size",
    "tsne_iterations": "t-SNE optimization iterations",
    "fastica_algorithm": "FastICA extraction algorithm",
    "fastica_fun": "FastICA contrast function",
    "fastica_max_iterations": "FastICA iterations",
    "umap_neighbors": "UMAP neighborhood size",
    "umap_min_dist": "UMAP minimum distance",
    "umap_metric": "UMAP distance metric",
}
HYPERPARAM_COLUMNS = list(HYPERPARAM_MEANING)
REDUCER_CRITERIA = [
    ("trustworthiness_mean", "Mean trustworthiness", "higher"),
    ("continuity_mean", "Mean continuity", "higher"),
    ("distance_spearman", "Spearman distance correlation", "higher"),
    ("quality_score", "Geometry quality score", "higher"),
]
CRITERION_LABELS = {
    "explained_variance_fraction": "Explained variance fraction",
    "normalized_stress": "Normalized MDS stress",
    "geodesic_reconstruction_error": "Isomap geodesic reconstruction error",
    "kl_divergence": "t-SNE KL divergence",
    "normalized_reconstruction_mse": "FastICA normalized reconstruction MSE",
    "geometry_quality_score": "Geometry quality score",
    "quality_score": "Geometry quality score",
}


def reducer_for(method, role):
    rows = selected_reducers[
        (selected_reducers["method"] == method)
        & selected_reducers["selection_role"].str.contains(role)
    ]
    if rows.empty:
        raise KeyError(f"No selected reducer for {method}/{role}")
    return rows.iloc[0]


def active_reducer_hyperparameters(candidates):
    return [column for column in HYPERPARAM_COLUMNS
            if candidates[column].notna().any()
            and candidates[column].nunique(dropna=True) > 1]


def parameter_envelope(ax, candidates, parameter, criterion, direction, selected_value):
    values = sorted(pd.unique(candidates[parameter].dropna()), key=str)
    envelope = []
    for value in values:
        scores = candidates.loc[candidates[parameter] == value, criterion]
        envelope.append(scores.max() if direction == "maximize" else scores.min())
    x = np.arange(len(values))
    ax.plot(x, envelope, "o-", color="#3366CC", linewidth=1.7, markersize=5)
    selected_index = next((i for i, value in enumerate(values)
                           if str(value) == str(selected_value)), None)
    if selected_index is not None:
        ax.scatter(selected_index, envelope[selected_index], marker="*", s=170,
                   color="#D55E00", edgecolor="#24292F", zorder=5)
    ax.set_xticks(x, [str(value) for value in values], rotation=30, ha="right")
    ax.set_xlabel(parameter.replace("_", " "))
    ax.set_ylabel(CRITERION_LABELS.get(criterion, criterion.replace("_", " ")))
    ax.grid(True, alpha=0.25)


for role in ROLE_NAMES:
    for method in REDUCTION_ORDER:
        winner = reducer_for(method, role)
        candidates = reduction_sweep[
            (reduction_sweep["method"] == method)
            & (reduction_sweep["n_components"] == int(winner["n_components"]))
            & (reduction_sweep["status"] == "ok")
        ].copy().sort_values("tag").reset_index(drop=True)
        if role == "latent_best":
            objective = str(winner["dimension_criterion"])
            candidates[objective] = candidates["dimension_criterion_value"]
            direction = str(winner["dimension_criterion_direction"])
        else:
            objective = "quality_score"
            direction = "maximize"

        hyperparameters = active_reducer_hyperparameters(candidates)
        panel_count = 1 + len(REDUCER_CRITERIA) + len(hyperparameters)
        columns = 3
        rows = int(np.ceil(panel_count / columns))
        fig, axes = plt.subplots(rows, columns, figsize=(5.2 * columns, 3.8 * rows),
                                 facecolor="white", constrained_layout=True)
        axes = np.atleast_1d(axes).ravel()
        winner_index = int(candidates.index[candidates["tag"] == winner["tag"]][0])

        def plot_candidate_curve(ax, x_values, y_values, winner_x, winner_y, star_size):
            # A single ok candidate (PCA at a fixed dimensionality has no other
            # hyperparameters to vary) must not render as a default-size circle
            # sitting directly under the winner star -- that combination is
            # visually indistinguishable from an empty panel.
            if len(x_values) == 1:
                ax.scatter(x_values, y_values, marker="D", s=90, color="#3366CC",
                          edgecolor="#24292F", linewidth=1.2, zorder=4,
                          label="single candidate")
                ax.legend(fontsize=6, frameon=False, loc="best")
            else:
                ax.plot(x_values, y_values, "o-", color="#3366CC",
                       linewidth=1.5, markersize=4)
            ax.scatter(winner_x, winner_y, marker="*", s=star_size,
                      color="#D55E00", edgecolor="#24292F", zorder=5)

        single_candidate = len(candidates) == 1
        suffix = " \u2014 single candidate" if single_candidate else ""
        if role == "latent_best":
            curve = dim_sweep[dim_sweep["method"] == method].sort_values("n_components")
            plot_candidate_curve(axes[0], curve["n_components"], curve["dimension_criterion_value"],
                                 winner["n_components"], winner["dimension_criterion_value"], 180)
            axes[0].set_xlabel("Number of dimensions")
            axes[0].set_ylabel(CRITERION_LABELS[objective])
            dim_suffix = " \u2014 single candidate" if len(curve) == 1 else ""
            axes[0].set_title(f"Dimension criterion{dim_suffix}")
        else:
            plot_candidate_curve(axes[0], np.arange(len(candidates)), candidates[objective],
                                 winner_index, winner[objective], 180)
            axes[0].set_xlabel("Candidate configuration")
            axes[0].set_ylabel(CRITERION_LABELS[objective])
            axes[0].set_title(f"Selection criterion{suffix}")
        axes[0].grid(True, alpha=0.25)

        # PCA has no reducer hyperparameters beyond n_components, so at any fixed
        # dimensionality it contributes exactly one candidate row here: plot_candidate_curve
        # renders that single row as an explicit diamond marker, not a broken multi-point curve.
        for ax, (column, label, preference) in zip(axes[1:], REDUCER_CRITERIA):
            plot_candidate_curve(ax, np.arange(len(candidates)), candidates[column],
                                 winner_index, winner[column], 160)
            ax.set_xlabel("Candidate configuration")
            ax.set_ylabel(label)
            ax.set_title(f"{label} ({preference}){suffix}")
            ax.grid(True, alpha=0.25)

        start = 1 + len(REDUCER_CRITERIA)
        for ax, parameter in zip(axes[start:], hyperparameters):
            parameter_envelope(ax, candidates, parameter, objective, direction,
                               winner[parameter])
        for ax in axes[panel_count:]:
            ax.remove()

        chosen = ", ".join(f"{p}={winner[p]}" for p in hyperparameters)
        fig.suptitle(
            f"{DISPLAY_NAMES[method]} reducer — {ROLE_NAMES[role]}\n"
            f"selected {int(winner['n_components'])}D"
            + (f"; {chosen}" if chosen else ""),
            fontsize=13, fontweight="bold",
        )
        fig.savefig(FIGURE_DIR / f"reducer_selection_{method}_{role}.png",
                    dpi=170, bbox_inches="tight", facecolor="white")
        plt.show()


##  Unclustered 2D and 3D embeddings



In [ ]:
INK = "#24292F"
GRID = "#E6E8EB"


def point_style(n_points, dimensions, noise=False):
    # Marker size scales inversely with element count so sparse and dense panels both stay readable.
    if dimensions == 2:
        size = float(np.clip(18000.0 / n_points, 0.55, 10.0))
    else:
        size = float(np.clip(14000.0 / n_points, 0.75, 8.0))
    alpha = 0.38 if n_points >= 20000 else 0.78
    if noise:
        return size * 0.8, 0.18 if n_points >= 20000 else 0.34
    return size, alpha


def format_embedding_axis(ax, dimensions):
    ax.set_xticks([])
    ax.set_yticks([])
    if dimensions == 3:
        ax.set_zticks([])
        ax.view_init(elev=22, azim=-58)
        ax.set_box_aspect((1, 1, 0.85))
        ax.grid(False)
    else:
        ax.set_aspect("equal", adjustable="datalim")
    for spine in getattr(ax, "spines", {}).values():
        spine.set_color(GRID)


def draw_unclustered(ax, coordinates, dimensions):
    size, alpha = point_style(len(coordinates), dimensions)
    if dimensions == 2:
        ax.scatter(coordinates[:, 0], coordinates[:, 1], s=size, color="#3366CC",
                   alpha=alpha, linewidths=0, rasterized=True)
    else:
        ax.scatter(coordinates[:, 0], coordinates[:, 1], coordinates[:, 2],
                   s=size, color="#3366CC", alpha=alpha, linewidths=0,
                   depthshade=False, rasterized=True)


for role, dimensions in (("display_2d", 2), ("display_3d", 3)):
    for method in REDUCTION_ORDER:
        reducer = reducer_for(method, role)
        coordinates = np.load(reducer["embedding"], mmap_mode="r")
        projection = {"projection": "3d"} if dimensions == 3 else {}
        fig = plt.figure(figsize=(6, 5.5), facecolor="white", constrained_layout=True)
        ax = fig.add_subplot(1, 1, 1, **projection)
        draw_unclustered(ax, coordinates, dimensions)
        format_embedding_axis(ax, dimensions)
        ax.set_title(
            f"{DISPLAY_NAMES[method]} — selected {dimensions}D embedding\n"
            f"all {len(coordinates):,} {ENTITY_PLURAL}; no clustering labels",
            color=INK, fontsize=12,
        )
        fig.savefig(FIGURE_DIR / f"embedding_{method}_{dimensions}d_all_{ENTITY_PLURAL}.png",
                    dpi=170, bbox_inches="tight", facecolor="white")
        plt.show()


## Clustering hyperparameter selection

2D, 3D, and latent-best roles are plotted separately for K-means, HDBSCAN, and BIRCH.


In [ ]:

import ast

CLUSTER_HYPERPARAM_MEANING = {
    "kmeans": ["n_clusters"],
    "hdbscan": ["min_cluster_size", "min_samples"],
    "birch": ["threshold", "branching_factor"],
}
CLUSTER_CRITERIA = [
    ("silhouette", "Silhouette coefficient", "higher"),
    ("davies_bouldin", "Davies–Bouldin index", "lower"),
    ("calinski_harabasz", "Calinski–Harabasz index", "higher"),
    ("cluster_balance", "Normalized cluster-size entropy", "higher"),
    ("assigned_fraction", "Assigned fraction", "higher"),
]


def cluster_balance(value):
    if not isinstance(value, str) or not value:
        return np.nan
    sizes = np.asarray(list(ast.literal_eval(value).values()), dtype=float)
    if len(sizes) < 2:
        return np.nan
    proportions = sizes / sizes.sum()
    return float(-(proportions * np.log(proportions)).sum() / np.log(len(sizes)))


cluster_sweep_metrics = cluster_sweep.copy()
cluster_sweep_metrics["cluster_balance"] = cluster_sweep_metrics[
    "cluster_sizes_raw_labels"
].map(cluster_balance)
cluster_sweep_metrics["assigned_fraction"] = 1.0 - cluster_sweep_metrics["noise_fraction"]


def selected_clustering(reducer_tag, clusterer):
    rows = selected_clusterings[
        (selected_clusterings["reducer_tag"] == reducer_tag)
        & (selected_clusterings["method"] == clusterer)
    ]
    return None if rows.empty else rows.iloc[0]


def plot_cluster_curve(ax, subset, primary, secondary, criterion, winner):
    if secondary is None:
        curve = subset.sort_values(primary)
        ax.plot(curve[primary], curve[criterion], "o-", color="#3366CC",
                linewidth=1.5, markersize=4)
    else:
        values = sorted(subset[secondary].dropna().unique())
        palette = matplotlib.colormaps["viridis"]
        for index, value in enumerate(values):
            curve = subset[subset[secondary] == value].sort_values(primary)
            ax.plot(curve[primary], curve[criterion], "o-",
                    color=palette(index / max(len(values) - 1, 1)),
                    linewidth=1.2, markersize=3,
                    label=f"{secondary}={value:g}")
    if criterion == "cluster_balance":
        winner_value = cluster_balance(winner["cluster_sizes_raw_labels"])
    elif criterion == "assigned_fraction":
        winner_value = 1.0 - winner["noise_fraction"]
    else:
        winner_value = winner[criterion]
    ax.scatter(winner[primary], winner_value, marker="*", s=140,
               color="#D55E00", edgecolor="#24292F", zorder=5)


# Filter candidates by fit success only; the pipeline's own winner-selection rule also
# enforces a winner cluster-count band, which NaNs out BIRCH/HDBSCAN candidates with
# well-defined individual criteria and would drop them from every panel below.
for role in ROLE_NAMES:
    for clusterer in CLUSTER_ORDER:
        parameters = CLUSTER_HYPERPARAM_MEANING[clusterer]
        primary = parameters[0]
        secondary = parameters[1] if len(parameters) > 1 else None
        fig, axes = plt.subplots(len(CLUSTER_CRITERIA), len(REDUCTION_ORDER),
                                 figsize=(22, 15), facecolor="white", constrained_layout=True)
        for column_index, method in enumerate(REDUCTION_ORDER):
            reducer = reducer_for(method, role)
            subset = cluster_sweep_metrics[
                (cluster_sweep_metrics["reducer_tag"] == reducer["tag"])
                & (cluster_sweep_metrics["method"] == clusterer)
                & (cluster_sweep_metrics["status"] == "ok")
            ]
            winner = selected_clustering(reducer["tag"], clusterer)
            for row_index, (criterion, label, preference) in enumerate(CLUSTER_CRITERIA):
                ax = axes[row_index, column_index]
                if subset.empty or winner is None:
                    ax.set_axis_off()
                    continue
                plot_cluster_curve(ax, subset, primary, secondary, criterion, winner)
                ax.set_xlabel(primary.replace("_", " "))
                ax.set_ylabel(label)
                ax.set_title(f"{DISPLAY_NAMES[method]} — {label} ({preference})", fontsize=9)
                ax.grid(True, alpha=0.22)
                if secondary is not None and row_index == 0:
                    ax.legend(fontsize=6, frameon=False)
        fig.suptitle(
            f"{CLUSTER_DISPLAY[clusterer]} criteria — {ROLE_NAMES[role]}\n"
            "orange star = the pipeline's selected winner",
            fontsize=14, fontweight="bold",
        )
        fig.savefig(FIGURE_DIR / f"cluster_selection_{clusterer}_{role}.png",
                    dpi=170, bbox_inches="tight", facecolor="white")
        plt.show()

winners_table = selected_clusterings.copy()
winners_table["cluster_balance"] = winners_table["cluster_sizes_raw_labels"].map(cluster_balance)
winners_table["assigned_fraction"] = 1.0 - winners_table["noise_fraction"]
display(winners_table[[
    "reduction_method", "selection_role", "method", "n_components", "n_clusters",
    "silhouette", "davies_bouldin", "calinski_harabasz", "cluster_balance",
    "assigned_fraction", "cluster_composite_score",
]].sort_values(["selection_role", "reduction_method", "method"]).reset_index(drop=True))


## Selected clustering solutions in 2D and 3D

All reducer × clusterer winners are plotted, with one color per cluster and gray for HDBSCAN noise.


In [ ]:
def _get_distinct_colormap(n_colors):
    colors = []
    for cmap_name in ("tab20", "tab20b", "tab20c"):
        cmap = matplotlib.colormaps[cmap_name]
        for index in range(cmap.N):
            if len(colors) >= n_colors:
                return colors
            colors.append(cmap(index / cmap.N))
    hsv = matplotlib.colormaps["hsv"]
    while len(colors) < n_colors:
        colors.append(hsv(len(colors) / n_colors))
    return colors


NOISE_COLOR = "#A8ADB5"


def find_map_record(reducer_tag, clusterer):
    return next((row for row in map_manifest
                 if row["reducer_tag"] == reducer_tag
                 and row["cluster_method"] == clusterer), None)


def setup_paths(method, role, clusterer):
    reducer = reducer_for(method, role)
    record = find_map_record(reducer["tag"], clusterer)
    if record is None:
        return reducer, None, None, None, None
    report_path = Path(record["report"])
    report = json.loads(report_path.read_text())
    return reducer, Path(reducer["embedding"]), report_path.parent / LABEL_FILENAME, report, record


def draw_clusters(ax, coordinates, labels, dimensions):
    labels = np.asarray(labels)
    cluster_labels = [int(value) for value in np.unique(labels) if value != -1]
    colors = dict(zip(cluster_labels, _get_distinct_colormap(len(cluster_labels))))
    ordered_labels = cluster_labels + ([-1] if np.any(labels == -1) else [])
    for raw_label in ordered_labels:
        mask = labels == raw_label
        size, alpha = point_style(len(labels), dimensions, noise=(raw_label == -1))
        color = NOISE_COLOR if raw_label == -1 else colors[int(raw_label)]
        if dimensions == 2:
            ax.scatter(coordinates[mask, 0], coordinates[mask, 1], s=size,
                       c=[color], alpha=alpha, linewidths=0, rasterized=True)
        else:
            ax.scatter(coordinates[mask, 0], coordinates[mask, 1], coordinates[mask, 2],
                       s=size, c=[color], alpha=alpha, linewidths=0,
                       depthshade=False, rasterized=True)


def plot_selected_display(method, clusterer, role, dimensions):
    reducer, embedding_path, labels_path, report, _ = setup_paths(method, role, clusterer)
    if labels_path is None:
        return
    coordinates = np.load(embedding_path, mmap_mode="r")
    labels = np.load(labels_path, mmap_mode="r")
    projection = {"projection": "3d"} if dimensions == 3 else {}
    fig = plt.figure(figsize=(6, 5.5), facecolor="white", constrained_layout=True)
    ax = fig.add_subplot(1, 1, 1, **projection)
    draw_clusters(ax, coordinates, labels, dimensions)
    format_embedding_axis(ax, dimensions)
    ax.set_title(
        f"{DISPLAY_NAMES[method]} × {CLUSTER_DISPLAY[clusterer]} — selected {dimensions}D\n"
        f"all {len(labels):,} {ENTITY_PLURAL}; {report['n_clusters']} clusters; "
        f"noise {report['noise_fraction']:.1%}",
        color=INK, fontsize=12,
    )
    fig.savefig(FIGURE_DIR / f"scatterplot_{method}_{clusterer}_{dimensions}d.png",
                dpi=170, bbox_inches="tight", facecolor="white")
    plt.show()


for role, dimensions in (("display_2d", 2), ("display_3d", 3)):
    for clusterer in CLUSTER_ORDER:
        for method in REDUCTION_ORDER:
            plot_selected_display(method, clusterer, role, dimensions)


##  Latent-best labels projected onto 2D and 3D embeddings

These panels cluster in each reducer's selected latent dimensionality, then show the saved labels on the corresponding display embedding.


In [ ]:
def latent_labels(method, clusterer):
    reducer, _, labels_path, report, record = setup_paths(method, "latent_best", clusterer)
    if labels_path is None:
        return None, None, None, None
    return reducer, np.asarray(np.load(labels_path, mmap_mode="r")), report, record


def plot_latent_projection(method, clusterer, role, dimensions):
    display_reducer, embedding_path, display_labels_path, display_report, _ = setup_paths(
        method, role, clusterer
    )
    latent_reducer, best_labels, latent_report, _ = latent_labels(method, clusterer)
    if display_labels_path is None or best_labels is None:
        return None
    coordinates = np.load(embedding_path, mmap_mode="r")
    display_labels = np.asarray(np.load(display_labels_path, mmap_mode="r"))
    ari = adjusted_rand_score(display_labels, best_labels)
    projection = {"projection": "3d"} if dimensions == 3 else {}
    fig = plt.figure(figsize=(6, 5.5), facecolor="white", constrained_layout=True)
    ax = fig.add_subplot(1, 1, 1, **projection)
    draw_clusters(ax, coordinates, best_labels, dimensions)
    format_embedding_axis(ax, dimensions)
    ax.set_title(
        f"{DISPLAY_NAMES[method]} × {CLUSTER_DISPLAY[clusterer]} latent-best labels\n"
        f"clustered in {int(latent_reducer['n_components'])}D, projected to {dimensions}D; "
        f"{latent_report['n_clusters']} clusters; ARI vs {dimensions}D fit={ari:.3f}",
        color=INK, fontsize=11,
    )
    fig.savefig(FIGURE_DIR / f"latent_labels_{method}_{clusterer}_on_{dimensions}d.png",
                dpi=170, bbox_inches="tight", facecolor="white")
    plt.show()
    return ari


latent_projection_records = []
for role, dimensions in (("display_2d", 2), ("display_3d", 3)):
    for clusterer in CLUSTER_ORDER:
        for method in REDUCTION_ORDER:
            ari = plot_latent_projection(method, clusterer, role, dimensions)
            if ari is not None:
                latent_projection_records.append({
                    "clusterer": clusterer, "reduction_method": method,
                    "display_role": role, "adjusted_rand_index": ari,
                })

display(pd.DataFrame(latent_projection_records))


## 6.  global-best channel labels


In [ ]:
latent_winners = selected_clusterings[
    selected_clusterings["selection_role"].str.contains("latent_best")
].copy()
best_winner = latent_winners.loc[latent_winners["cluster_composite_score"].idxmax()]
best_record = find_map_record(best_winner["reducer_tag"], best_winner["method"])
source_report = Path(best_record["report"])
source_dir = source_report.parent

best_outputs = {
    "labels_npy": SELECTION_ROOT / "best_channel_clustering.npy",
    "labels_csv": SELECTION_ROOT / "best_channel_clustering.csv",
    "manifest": SELECTION_ROOT / "best_channel_clustering.json",
}
source_files = {
    "labels_npy": source_dir / "channel_labels.npy",
    "labels_csv": source_dir / "channel_labels.csv",
}
for key, source in source_files.items():
    target = best_outputs[key]
    if (not target.exists()) or source.stat().st_mtime_ns > target.stat().st_mtime_ns:
        shutil.copy2(source, target)

best_manifest = {
    "selection_scope": "global maximum cluster_composite_score among latent_best reducer × clusterer winners",
    "family": FAMILY,
    "reduction_method": str(best_winner["reduction_method"]),
    "reducer_tag": str(best_winner["reducer_tag"]),
    "cluster_method": str(best_winner["method"]),
    "cluster_tag": str(best_winner["full_fit_cluster_tag"]),
    "n_components": int(best_winner["n_components"]),
    "n_clusters": int(best_winner["n_clusters"]),
    "cluster_composite_score": float(best_winner["cluster_composite_score"]),
    "source_report": str(source_report),
    "labels_npy": str(best_outputs["labels_npy"]),
    "labels_csv": str(best_outputs["labels_csv"]),
}
best_outputs["manifest"].write_text(json.dumps(best_manifest, indent=2, sort_keys=True) + "\n")

best_labels = np.load(best_outputs["labels_npy"])
assert len(best_labels) == N_ELEMENTS
print(f"{CLUSTER_DISPLAY[best_manifest['cluster_method']]} on {DISPLAY_NAMES[best_manifest['reduction_method']]} "
      f"({best_manifest['n_components']}D latent, {best_manifest['n_clusters']} clusters, "
      f"composite score {best_manifest['cluster_composite_score']:.3f})")
saved_to(SELECTION_ROOT)


## Global-best labels on 2D and 3D latent spaces


In [ ]:
best_method = str(best_winner["reduction_method"])
best_clusterer = str(best_winner["method"])
for role, dimensions in (("display_2d", 2), ("display_3d", 3)):
    display_reducer = reducer_for(best_method, role)
    coordinates = np.load(display_reducer["embedding"], mmap_mode="r")
    projection = {"projection": "3d"} if dimensions == 3 else {}
    fig = plt.figure(figsize=(6, 5.5), facecolor="white", constrained_layout=True)
    ax = fig.add_subplot(1, 1, 1, **projection)
    draw_clusters(ax, coordinates, best_labels, dimensions)
    format_embedding_axis(ax, dimensions)
    ax.set_title(
        f"Global best latent clustering on {dimensions}D {DISPLAY_NAMES[best_method]} space\n"
        f"{CLUSTER_DISPLAY[best_clusterer]}; clustered in {int(best_winner['n_components'])}D; "
        f"all {len(best_labels):,} {ENTITY_PLURAL}",
        color=INK, fontsize=11,
    )
    fig.savefig(FIGURE_DIR / f"best_{ENTITY_SINGULAR}_clustering_on_{dimensions}d.png",
                dpi=170, bbox_inches="tight", facecolor="white")
    plt.show()


## cluster differentiation

In [ ]:
screen_path = OUTPUTS / "cluster/_channel_vertex_alignment_screen.csv"
screen = pd.read_csv(screen_path)
evidence = screen[(screen["side"] == "channel") & (screen["family"] == FAMILY)].copy()
evidence = evidence.merge(
    winners_table[["reducer_tag", "cluster_tag", "method", "silhouette"]],
    left_on=["reducer_tag", "cluster_tag", "cluster_method"],
    right_on=["reducer_tag", "cluster_tag", "method"], how="left",
)

fig, ax = plt.subplots(figsize=(7, 5.5), facecolor="white", constrained_layout=True)
for clusterer, marker in {"kmeans": "o", "hdbscan": "s", "birch": "^"}.items():
    rows = evidence[evidence["cluster_method"] == clusterer]
    ax.scatter(rows["silhouette"], rows["mean_abs_off_diag"], marker=marker,
               label=CLUSTER_DISPLAY[clusterer], alpha=0.75)
ax.set_xlabel("Silhouette coefficient")
ax.set_ylabel("Mean absolute correlation between cluster timeseries")
ax.set_title(f"{FAMILY} selected solutions: separation vs temporal redundancy")
ax.legend(frameon=False)
ax.grid(True, alpha=0.25)
fig.savefig(FIGURE_DIR / "cluster_evidence_internal_vs_temporal.png",
            dpi=170, bbox_inches="tight", facecolor="white")
plt.show()

solutions_table = winners_table.merge(
    evidence[["reducer_tag", "cluster_tag", "cluster_method", "mean_abs_off_diag"]],
    left_on=["reducer_tag", "cluster_tag", "method"],
    right_on=["reducer_tag", "cluster_tag", "cluster_method"], how="left",
)
display(solutions_table[[
    "selection_role", "reduction_method", "reducer_tag", "n_components",
    "method", "cluster_tag", "n_clusters",
    "silhouette", "davies_bouldin", "calinski_harabasz", "mean_abs_off_diag",
    "cluster_composite_score",
]].sort_values(["selection_role", "reduction_method", "method"]).reset_index(drop=True))
